In [1]:
import numpy as np
import pandas as pd


In [2]:
np.random.seed(42)


In [3]:
def calculate_health_score(
    task_completion,
    attendance,
    grade,
    focus_hours,
    streak,
    overdue_ratio
):
    score = 0

    # Task completion (strong signal)
    score += task_completion * 30

    # Attendance (strong signal)
    score += (attendance / 100) * 25

    # Grades
    score += (grade / 10) * 20

    # Focus behavior
    score += min(focus_hours / 20, 1) * 15

    # Consistency
    score += streak * 10

    # Penalty for overdue behavior
    score -= overdue_ratio * 15

    # Small human noise
    score += np.random.uniform(-3, 3)

    return max(0, min(100, score))


In [4]:
rows = []
NUM_SAMPLES = 80000

for _ in range(NUM_SAMPLES):
    task_completion = np.random.uniform(0.3, 1.0)
    attendance = np.random.uniform(60, 100)
    grade = np.random.uniform(4, 10)
    focus_hours = np.random.uniform(0, 35)
    streak = np.random.uniform(0.2, 1.0)
    overdue_ratio = np.random.uniform(0, 0.5)

    health = calculate_health_score(
        task_completion,
        attendance,
        grade,
        focus_hours,
        streak,
        overdue_ratio
    )

    rows.append([
        task_completion,
        attendance,
        grade,
        focus_hours,
        streak,
        overdue_ratio,
        health
    ])


In [5]:
columns = [
    "task_completion_rate",
    "attendance_percent",
    "average_grade",
    "focus_hours_week",
    "streak_consistency",
    "overdue_ratio",
    "academic_health"
]

df_health = pd.DataFrame(rows, columns=columns)
df_health.head()


,task_completion_rate,attendance_percent,average_grade,focus_hours_week,streak_consistency,overdue_ratio,academic_health
0,0.562178,98.028572,8.391964,20.953047,0.324815,0.077997,72.583105
1,0.906323,84.044600,8.248435,0.720457,0.975928,0.416221,67.028057
2,0.427277,67.336180,5.825453,18.366475,0.545556,0.145615,59.020592
3,0.397646,71.685786,6.198171,15.962449,0.828141,0.099837,61.088258
4,0.714690,61.858017,7.645269,5.968344,0.252041,0.474443,54.869570


In [6]:
df_health.to_csv("acadbox_academic_health_data.csv", index=False)
print("Academic Health dataset saved.")


Academic Health dataset saved.


In [7]:
import pandas as pd
import numpy as np

from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, r2_score

import joblib


In [8]:
df = pd.read_csv("acadbox_academic_health_data.csv")

print("Dataset shape:", df.shape)
df.head()


Dataset shape: (80000, 7)


,task_completion_rate,attendance_percent,average_grade,focus_hours_week,streak_consistency,overdue_ratio,academic_health
0,0.562178,98.028572,8.391964,20.953047,0.324815,0.077997,72.583105
1,0.906323,84.044600,8.248435,0.720457,0.975928,0.416221,67.028057
2,0.427277,67.336180,5.825453,18.366475,0.545556,0.145615,59.020592
3,0.397646,71.685786,6.198171,15.962449,0.828141,0.099837,61.088258
4,0.714690,61.858017,7.645269,5.968344,0.252041,0.474443,54.869570


In [9]:
X = df.drop("academic_health", axis=1)
y = df["academic_health"]

print("Features:", list(X.columns))


Features: ['task_completion_rate', 'attendance_percent', 'average_grade', 'focus_hours_week', 'streak_consistency', 'overdue_ratio']


In [10]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)


In [11]:
model = RandomForestRegressor(
    n_estimators=300,
    max_depth=18,
    min_samples_split=4,
    min_samples_leaf=2,
    random_state=42,
    n_jobs=-1
)


In [12]:
model.fit(X_train, y_train)
print("Academic Health model trained.")


Academic Health model trained.


In [13]:
y_pred = model.predict(X_test)

mae = mean_absolute_error(y_test, y_pred)
r2 = r2_score(y_test, y_pred)

print("Mean Absolute Error (MAE):", round(mae, 2))
print("R² Score:", round(r2, 3))


Mean Absolute Error (MAE): 1.67
R² Score: 0.958


In [16]:
joblib.dump(model, "acadbox_academic_health_model.pkl")
print("Academic Health model saved.")


Academic Health model saved.


In [17]:
sample = [[
    0.85,   # task_completion_rate
    92,     # attendance_percent
    8.1,    # average_grade
    14,     # focus_hours_week
    0.8,    # streak_consistency
    0.1     # overdue_ratio
]]

predicted_health = model.predict(sample)[0]
print("Predicted Academic Health:", round(predicted_health, 1))


Predicted Academic Health: 82.2


/home/punit/anaconda3/lib/python3.13/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but RandomForestRegressor was fitted with feature names
  warnings.warn(
